# Notebook 2: Retrieval Evaluation

Evaluate the retrieval layer in isolation — before the LLM ever sees the context. If retrieval is broken, no amount of prompt engineering will fix the final answer.

## Setup

In [1]:
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI
import chromadb

load_dotenv()
client          = OpenAI()
EMBEDDING_MODEL = "text-embedding-3-small"
DATA_PATH       = Path("../data/AI_Agent_Insure.md")
GOLDEN_PATH     = Path("../data/golden_dataset.json")

## Part 1: Load the golden dataset and build the index

In [2]:
golden = json.loads(GOLDEN_PATH.read_text(encoding="utf-8"))

# Only questions with relevant_chunk_ids filled in are useful for retrieval eval.
retrieval_cases = [e for e in golden if e["relevant_chunk_ids"]]
print(f"Total golden entries  : {len(golden)}")
print(f"Retrieval eval cases  : {len(retrieval_cases)}")
print()
print("Note: if relevant_chunk_ids are empty, run Notebook 1 first and fill them in.")

Total golden entries  : 10
Retrieval eval cases  : 10

Note: if relevant_chunk_ids are empty, run Notebook 1 first and fill them in.


In [3]:
# Sentence chunking
def load_and_chunk(path: Path) -> list[str]:
    doc    = path.read_text(encoding="utf-8").strip()
    chunks = [s.strip() for s in doc.replace("\n", " ").split(".") if s.strip()]
    return [c + "." for c in chunks]


chunks     = load_and_chunk(DATA_PATH)
resp       = client.embeddings.create(model=EMBEDDING_MODEL, input=chunks)
embeddings = [r.embedding for r in resp.data]

chroma = chromadb.EphemeralClient()
coll   = chroma.create_collection("evals_retrieval")
coll.add(
    ids=[str(i) for i in range(len(chunks))],
    embeddings=embeddings,
    documents=chunks
)
print(f"Index built: {len(chunks)} chunks")

Index built: 34 chunks


## Part 2: The three retrieval metrics

**Precision@k** — of the top-k chunks returned, what fraction are actually relevant?
$$\text{Precision@k} = \frac{|\text{retrieved} \cap \text{relevant}|}{k}$$

**Recall@k** — of all relevant chunks, what fraction did we retrieve in the top-k?
$$\text{Recall@k} = \frac{|\text{retrieved} \cap \text{relevant}|}{|\text{relevant}|}$$

**MRR (Mean Reciprocal Rank)** — how high up is the *first* relevant chunk?
$$\text{MRR} = \frac{1}{|Q|} \sum_{q} \frac{1}{\text{rank of first relevant chunk}}$$

MRR is often the most useful single number: if the first relevant chunk is at rank 1, score = 1.0. At rank 2, score = 0.5. At rank 5, score = 0.2.

In [4]:
def precision_at_k(retrieved_ids: list[str], relevant_ids: list[str], k: int) -> float:
    retrieved_k = set(retrieved_ids[:k])
    relevant    = set(str(i) for i in relevant_ids)
    hits        = retrieved_k & relevant
    return len(hits) / k if k > 0 else 0.0


def recall_at_k(retrieved_ids: list[str], relevant_ids: list[str], k: int) -> float:
    retrieved_k = set(retrieved_ids[:k])
    relevant    = set(str(i) for i in relevant_ids)
    if not relevant:
        return 0.0
    hits = retrieved_k & relevant
    return len(hits) / len(relevant)


def reciprocal_rank(retrieved_ids: list[str], relevant_ids: list[str]) -> float:
    relevant = set(str(i) for i in relevant_ids)
    for rank, doc_id in enumerate(retrieved_ids, start=1):
        if doc_id in relevant:
            return 1.0 / rank
    return 0.0


print("Metric functions defined.")

Metric functions defined.


## Part 3: Run retrieval eval at k=8

In [5]:
def retrieve(query: str, k: int = 8) -> list[str]:
    """Embed query, return top-k chunk IDs from ChromaDB."""
    q_emb    = client.embeddings.create(model=EMBEDDING_MODEL, input=[query])
    q_vec    = q_emb.data[0].embedding
    results  = coll.query(query_embeddings=[q_vec], n_results=k)
    return results["ids"][0]


K = 8
results_table = []

for case in retrieval_cases:
    retrieved = retrieve(case["question"], k=K)
    p         = precision_at_k(retrieved, case["relevant_chunk_ids"], K)
    r         = recall_at_k(retrieved, case["relevant_chunk_ids"], K)
    rr        = reciprocal_rank(retrieved, case["relevant_chunk_ids"])

    results_table.append({
        "id":          case["id"],
        "question":    case["question"][:60],
        "precision@8": round(p, 2),
        "recall@8":    round(r, 2),
        "rr":          round(rr, 2),
        "retrieved":   retrieved,
        "relevant":    [str(i) for i in case["relevant_chunk_ids"]],
    })

print(f"{'ID':>4}  {'P@8':>5}  {'R@8':>5}  {'RR':>5}  Question")
print("-" * 80)
for row in results_table:
    print(f"{row['id']:>4}  {row['precision@8']:>5}  {row['recall@8']:>5}  {row['rr']:>5}  {row['question']}")

  ID    P@8    R@8     RR  Question
--------------------------------------------------------------------------------
 q01   0.12    1.0    1.0  What does Agentic AI Liability Insurance cover?
 q02   0.12    1.0    1.0  How much would Model & Data Security Insurance cost for a st
 q03    0.0    0.0    0.0  Can a healthcare company get coverage from AI Agent Insure?
 q04    0.0    0.0    0.0  How does AI Agent Insure handle claims?
 q05   0.12    0.5    1.0  What is AI Agent Insure's underwriting philosophy?
 q06   0.25    1.0    1.0  What does Autonomous Systems & Robotics Coverage include?
 q07   0.12    1.0    1.0  How much does Agentic AI Liability Insurance cost for an ent
 q08    0.0    0.0    0.0  Is a synthetic data company eligible for coverage?
 q09   0.25    1.0    1.0  I'm a robotics startup. Am I eligible and what would coverag
 q10   0.25    1.0    1.0  Explain the claims process and what Model & Data Security In


## Part 4: Aggregate scores

In [6]:
if results_table:
    mean_p   = sum(r["precision@8"] for r in results_table) / len(results_table)
    mean_r   = sum(r["recall@8"]    for r in results_table) / len(results_table)
    mean_mrr = sum(r["rr"]          for r in results_table) / len(results_table)

    print("Aggregate retrieval scores")
    print("-" * 30)
    print(f"Mean Precision@8 : {mean_p:.3f}")
    print(f"Mean Recall@8    : {mean_r:.3f}")
    print(f"MRR              : {mean_mrr:.3f}")
else:
    print("No retrieval cases to score — fill in relevant_chunk_ids in the golden dataset.")

Aggregate retrieval scores
------------------------------
Mean Precision@8 : 0.123
Mean Recall@8    : 0.650
MRR              : 0.700


## Part 5: Compare two chunking strategies

One of the main uses of retrieval eval: compare strategies objectively. Here we compare sentence chunking (used throughout the series) against a fixed token-window approach.

In [7]:
def chunk_by_tokens(path: Path, window: int = 50, overlap: int = 10) -> list[str]:
    """Split document into fixed-size token windows with overlap."""
    doc    = path.read_text(encoding="utf-8").strip()
    words  = doc.split()
    chunks = []
    step   = window - overlap
    for i in range(0, len(words), step):
        chunk = " ".join(words[i : i + window])
        if chunk:
            chunks.append(chunk)
    return chunks


token_chunks = chunk_by_tokens(DATA_PATH, window=50, overlap=10)
print(f"Sentence chunks : {len(chunks)}")
print(f"Token chunks    : {len(token_chunks)}")

# Build a second index with token chunks.
resp2       = client.embeddings.create(model=EMBEDDING_MODEL, input=token_chunks)
embeddings2 = [r.embedding for r in resp2.data]

coll2 = chroma.create_collection("evals_retrieval_token")
coll2.add(
    ids=[str(i) for i in range(len(token_chunks))],
    embeddings=embeddings2,
    documents=token_chunks
)
print("Token-chunk index ready.")

Sentence chunks : 34
Token chunks    : 18
Token-chunk index ready.


In [8]:
# For questions that have relevant_chunk_ids, we can only compare on the
# sentence-chunked index (since the IDs don't map across strategies).
# Instead, we compare qualitatively: look at what each strategy retrieves
# for the same question.

if retrieval_cases:
    sample = retrieval_cases[0]
    q      = sample["question"]

    q_emb = client.embeddings.create(model=EMBEDDING_MODEL, input=[q])
    q_vec = q_emb.data[0].embedding

    sent_results  = coll.query( query_embeddings=[q_vec], n_results=3)
    token_results = coll2.query(query_embeddings=[q_vec], n_results=3)

    print(f"Question: {q}")
    print()
    print("--- Sentence chunks (top 3) ---")
    for doc in sent_results["documents"][0]:
        print(f"  {doc[:120]}")
    print()
    print("--- Token window chunks (top 3) ---")
    for doc in token_results["documents"][0]:
        print(f"  {doc[:120]}")
else:
    print("Fill in relevant_chunk_ids in the golden dataset to enable comparison.")

Question: What does Agentic AI Liability Insurance cover?

--- Sentence chunks (top 3) ---
  Agentic AI Liability Insurance 3.
  AI Agent Insure provides tailored insurance solutions addressing operational, regulatory, financial, physical, and reput
  Company Overview  AI Agent Insure is a specialty insurance organization focused exclusively on risks arising from artifi

--- Token window chunks (top 3) ---
  2. Agentic AI Liability Insurance 3. Autonomous Systems & Robotics Coverage 4. Compliance & Regulatory Shield 5. Agentic
  agents, machine learning infrastructure, and AI-driven decision-making platforms. The company is designed as an AI-nativ
  Systems & Robotics Coverage - Compliance & Regulatory Shield - Agentic Workflow Uptime Insurance - Intellectual Property


## Part 6: What the scores mean

| Score | What it tells you |
|---|---|
| Precision@k = 1.0 | Every retrieved chunk is relevant — no noise |
| Recall@k = 1.0 | Every relevant chunk was retrieved — nothing missed |
| MRR = 1.0 | The first retrieved chunk is always relevant |
| MRR < 0.3 | Relevant content is consistently buried — retrieval needs work |

In practice, precision and recall trade off. A larger k improves recall but hurts precision (and increases LLM context cost). The right k depends on your use case.

## Key concepts

- Retrieval can be evaluated without the LLM — it's just set intersection math
- Precision@k, Recall@k, and MRR each measure a different aspect of retrieval quality
- Use retrieval eval to compare chunking strategies, embedding models, or k values objectively
- **Next:** Notebook 3 evaluates the generation layer — what happens after the chunks are retrieved